# llm-join — Test Notebook

No API key needed. Uses:
- **Mock LLM** — scores candidates using word overlap + character similarity heuristics
- **Mock embed_fn** — TF-IDF style character n-gram vectors (numpy only, no sklearn)

Swap the mock functions for your real LLM/embed when ready.

## 1. Install

In [ ]:
# Install from local clone (run from repo root) or GitHub
# pip install -e ..
# pip install git+https://github.com/adityabalki/llm-join.git

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # works if notebook is in /notebooks/

## 2. Mock LLM and embed_fn (no API key needed)

In [ ]:
import json
import re
import numpy as np


# ── Mock embed_fn ────────────────────────────────────────────────────────────
# Character n-gram (trigram) bag-of-words vectors.
# Good enough for faiss to retrieve plausible candidates.

def get_trigrams(text: str) -> set:
    t = text.lower()
    return {t[i:i+3] for i in range(len(t) - 2)}

def mock_embed(texts: list[str]) -> np.ndarray:
    """Returns (n, 300) float32 array using hashed trigram features."""
    dim = 300
    vecs = np.zeros((len(texts), dim), dtype="float32")
    for i, text in enumerate(texts):
        for tg in get_trigrams(text):
            idx = hash(tg) % dim
            vecs[i, idx] += 1.0
        norm = np.linalg.norm(vecs[i])
        if norm > 0:
            vecs[i] /= norm
    return vecs


# ── Mock LLM ─────────────────────────────────────────────────────────────────
# Parses the prompt, scores each LEFT|RIGHT pair using:
#   - word overlap (Jaccard on lowercase words)
#   - character trigram overlap
# Returns valid JSON the scorer can parse.

def word_jaccard(a: str, b: str) -> float:
    wa = set(re.sub(r'[^a-z0-9 ]', '', a.lower()).split())
    wb = set(re.sub(r'[^a-z0-9 ]', '', b.lower()).split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def trigram_overlap(a: str, b: str) -> float:
    ta, tb = get_trigrams(a), get_trigrams(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

def mock_llm(prompt: str) -> str:
    """Parses llm-join prompt format and returns JSON scores."""
    left_val = ""
    candidates = []

    for line in prompt.splitlines():
        m = re.match(r'(\d+)\. LEFT: "(.+?)" \| RIGHT: "(.+?)"', line)
        if m:
            if not left_val:
                left_val = m.group(2)
            candidates.append((int(m.group(1)), m.group(2), m.group(3)))

    results = []
    for idx, left, right in candidates:
        wj = word_jaccard(left, right)
        tg = trigram_overlap(left, right)
        score = round(0.5 * wj + 0.5 * tg, 3)
        if score > 0.6:
            reasoning = "strong word and character overlap"
        elif score > 0.3:
            reasoning = "partial match"
        else:
            reasoning = "low similarity"
        results.append({"index": idx, "score": score, "reasoning": reasoning})

    return json.dumps(results)


# Quick sanity check
print(mock_embed(["Goldman Sachs", "Amazon"]).shape)   # (2, 300)
print(mock_llm('0. LEFT: "aspirin" | RIGHT: "Bayer Aspirin"'))

## 3. Test Data

In [ ]:
import pandas as pd

# Left: your internal vendor list
df1 = pd.DataFrame({
    "vendor":  ["Goldman Sachs & Co.", "Amazon Web Services", "Microsoft Corp",
                "Apple Inc", "JP Morgan", "Unmatched Vendor Ltd"],
    "spend":   [1_200_000, 890_000, 340_000, 500_000, 220_000, 99_000],
})

# Right: supplier master (different naming conventions)
df2 = pd.DataFrame({
    "supplier_name": ["The Goldman Sachs Group Inc", "Amazon.com Inc.",
                      "Microsoft Corporation", "Apple Incorporated",
                      "JPMorgan Chase & Co.", "Extra Supplier Not In Left"],
    "category":      ["Finance", "Cloud", "Software", "Hardware", "Finance", "Other"],
})

print("df1:"); display(df1)
print("df2:"); display(df2)

## 4. Basic Inner Join

In [ ]:
from llm_join import fuzzy_join

result = fuzzy_join(
    df1, df2,
    left_on="vendor",
    right_on="supplier_name",
    llm=mock_llm,
    embed_fn=mock_embed,
    context="company vendor names — match legal entity variants and abbreviations",
    how="inner",
    threshold=0.3,   # lower threshold for mock (real LLM scores higher)
)

display(result)

## 5. Left Join — See What Didn't Match

In [ ]:
result_left = fuzzy_join(
    df1, df2,
    left_on="vendor",
    right_on="supplier_name",
    llm=mock_llm,
    embed_fn=mock_embed,
    how="left",
    threshold=0.3,
)

display(result_left)

unmatched = result_left[result_left["supplier_name"].isna()]
print(f"\nUnmatched left rows ({len(unmatched)}):")
display(unmatched)

## 6. Return Reasoning — See Why Each Match Was Made

In [ ]:
result_reasoned = fuzzy_join(
    df1, df2,
    left_on="vendor",
    right_on="supplier_name",
    llm=mock_llm,
    embed_fn=mock_embed,
    how="left",
    threshold=0.3,
    return_reasoning=True,
)

display(result_reasoned[["vendor", "supplier_name", "_llm_score", "_llm_reasoning", "_embed_rank", "_match_method"]])

## 7. Outer Join — Full Reconciliation (both gaps)

In [ ]:
result_outer = fuzzy_join(
    df1, df2,
    left_on="vendor",
    right_on="supplier_name",
    llm=mock_llm,
    embed_fn=mock_embed,
    how="outer",
    threshold=0.3,
)

display(result_outer)

# Rows in supplier master with no match in your vendor list
unmatched_right = result_outer[result_outer["vendor"].isna()]
print(f"\nRight rows with no left match ({len(unmatched_right)}):")
display(unmatched_right)

## 8. Cost Controls

In [ ]:
import warnings

result_capped = fuzzy_join(
    df1, df2,
    left_on="vendor",
    right_on="supplier_name",
    llm=mock_llm,
    embed_fn=mock_embed,
    top_k=3,              # fewer candidates per row
    threshold=0.3,
    max_llm_calls=3,      # cap at 3 LLM calls (will warn and return partial)
    how="left",
)

display(result_capped)

## 9. Chained Joins

In [ ]:
# df1: transactions with vendor + product
transactions = pd.DataFrame({
    "vendor":  ["Goldman Sachs & Co.", "Microsoft Corp"],
    "product": ["equity trading platform", "office suite software"],
    "amount":  [50_000, 12_000],
})

product_catalog = pd.DataFrame({
    "catalog_item": ["Equity Trading System", "Microsoft Office 365", "Azure Cloud"],
    "sku":          ["EQ-001", "MS-OFF-365", "AZ-001"],
})

# Step 1: match vendors
step1 = fuzzy_join(
    transactions, df2,
    left_on="vendor", right_on="supplier_name",
    llm=mock_llm, embed_fn=mock_embed,
    how="left", threshold=0.3,
    return_reasoning=True,
)
step1 = step1.rename(columns={
    "_llm_score": "_vendor_score",
    "_llm_reasoning": "_vendor_reasoning",
    "_match_method": "_vendor_method",
    "_embed_rank": "_vendor_embed_rank",
})

# Step 2: match products
result_chained = fuzzy_join(
    step1, product_catalog,
    left_on="product", right_on="catalog_item",
    llm=mock_llm, embed_fn=mock_embed,
    how="left", threshold=0.1,
    return_reasoning=True,
)

display(result_chained)

## 10. Swap to Real LLM / Embed

When you have API access, replace the mock functions:

```python
import openai, numpy as np
client = openai.OpenAI(api_key="...")  # or base_url="https://your-enterprise-endpoint"

def my_llm(prompt):
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    ).choices[0].message.content

def my_embed(texts):
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([d.embedding for d in response.data], dtype="float32")

# Then use threshold=0.7 (default) instead of 0.3
result = fuzzy_join(df1, df2, left_on="vendor", right_on="supplier_name",
                    llm=my_llm, embed_fn=my_embed)
```